In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split

df = pd.read_csv("../data/churn_clean.csv")
print(df.shape)

X = df.drop(columns=['Churn'])
y = df['Churn']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(X_train.shape, X_test.shape)
print(y_train.mean(), y_test.mean())

(7043, 19)
(5634, 18) (1409, 18)
0.2653532126375577 0.2654364797728886


In [2]:
print(X_train.dtypes)

SeniorCitizen         int64
Partner                 str
Dependents              str
tenure                int64
PhoneService            str
MultipleLines           str
InternetService         str
OnlineSecurity          str
OnlineBackup            str
DeviceProtection        str
TechSupport             str
StreamingTV             str
StreamingMovies         str
Contract                str
PaperlessBilling        str
PaymentMethod           str
MonthlyCharges      float64
TotalCharges        float64
dtype: object


In [3]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

In [6]:
num_cols = ['tenure', 'MonthlyCharges', 'TotalCharges']
cat_cols = ['SeniorCitizen', 'Partner', 'Dependents', 'PhoneService',
            'MultipleLines', 'InternetService', 'OnlineSecurity',
            'OnlineBackup', 'DeviceProtection', 'TechSupport',
            'StreamingTV', 'StreamingMovies', 'Contract',
            'PaperlessBilling', 'PaymentMethod']

In [7]:
preprocessor = ColumnTransformer([
    ('num', StandardScaler(), num_cols),
    ('cat', OneHotEncoder(drop='if_binary', handle_unknown='ignore',
                          sparse_output=False), cat_cols),
])


In [8]:
X_train_prep = preprocessor.fit_transform(X_train)
X_test_prep = preprocessor.transform(X_test)

print(X_train_prep.shape, X_test_prep.shape)

(5634, 25) (1409, 25)


In [10]:
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.dummy import DummyClassifier
from sklearn.model_selection import StratifiedKFold, cross_validate

In [11]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scoring = ['accuracy', 'precision', 'recall', 'f1', 'roc_auc']

# 1. "Dumb" baseline: always predicts the majority class (No churn)
dummy = Pipeline([('prep', preprocessor),
                  ('model', DummyClassifier(strategy='most_frequent'))])

# 2. Logistic regression baseline
logreg = Pipeline([('prep', preprocessor),
                   ('model', LogisticRegression(max_iter=1000))])

for name, pipe in [('Dummy', dummy), ('LogReg', logreg)]:
    res = cross_validate(pipe, X_train, y_train, cv=cv, scoring=scoring)
    print(name)
    for m in scoring:
        print(f"  {m:10s} {res['test_' + m].mean():.3f}")

/Users/mac/Desktop/churn-prediction /venv/lib/python3.14/site-packages/sklearn/metrics/_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/Users/mac/Desktop/churn-prediction /venv/lib/python3.14/site-packages/sklearn/metrics/_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/Users/mac/Desktop/churn-prediction /venv/lib/python3.14/site-packages/sklearn/metrics/_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capi

Dummy
  accuracy   0.735
  precision  0.000
  recall     0.000
  f1         0.000
  roc_auc    0.500
LogReg
  accuracy   0.803
  precision  0.655
  recall     0.545
  f1         0.595
  roc_auc    0.846


In [12]:
logreg_bal = Pipeline([('prep', preprocessor),
                       ('model', LogisticRegression(max_iter=1000,
                                                    class_weight='balanced'))])

res = cross_validate(logreg_bal, X_train, y_train, cv=cv, scoring=scoring)
print('LogReg balanced')
for m in scoring:
    print(f"  {m:10s} {res['test_' + m].mean():.3f}")

LogReg balanced
  accuracy   0.750
  precision  0.519
  recall     0.804
  f1         0.631
  roc_auc    0.846


In [13]:
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier

neg, pos = (y_train == 0).sum(), (y_train == 1).sum()

models = {
    'RandomForest': RandomForestClassifier(n_estimators=300, class_weight='balanced',
                                           random_state=42, n_jobs=-1),
    'XGBoost': XGBClassifier(n_estimators=300, learning_rate=0.05, max_depth=3,
                             scale_pos_weight=neg/pos, random_state=42,
                             eval_metric='logloss'),
}

for name, model in models.items():
    pipe = Pipeline([('prep', preprocessor), ('model', model)])
    res = cross_validate(pipe, X_train, y_train, cv=cv, scoring=scoring)
    print(name)
    for m in scoring:
        print(f"  {m:10s} {res['test_' + m].mean():.3f}")

RandomForest
  accuracy   0.780
  precision  0.575
  recall     0.656
  f1         0.613
  roc_auc    0.827
XGBoost
  accuracy   0.753
  precision  0.522
  recall     0.798
  f1         0.631
  roc_auc    0.845


In [14]:
from sklearn.model_selection import GridSearchCV, RandomizedSearchCV


In [15]:
# Tune logistic regression
lr_pipe = Pipeline([('prep', preprocessor),
                    ('model', LogisticRegression(max_iter=1000, class_weight='balanced'))])
lr_search = GridSearchCV(lr_pipe, {'model__C': [0.01, 0.1, 1, 10]},
                         cv=cv, scoring='roc_auc')
lr_search.fit(X_train, y_train)
print('LogReg best:', lr_search.best_params_, round(lr_search.best_score_, 4))

LogReg best: {'model__C': 10} 0.8463


In [16]:
# Tune XGBoost
xgb_pipe = Pipeline([('prep', preprocessor),
                     ('model', XGBClassifier(scale_pos_weight=neg/pos, random_state=42,
                                             eval_metric='logloss'))])
xgb_params = {
    'model__n_estimators': [100, 200, 300],
    'model__max_depth': [2, 3, 4],
    'model__learning_rate': [0.01, 0.03, 0.05, 0.1],
    'model__subsample': [0.7, 0.85, 1.0],
}
xgb_search = RandomizedSearchCV(xgb_pipe, xgb_params, n_iter=20, cv=cv,
                                scoring='roc_auc', random_state=42)
xgb_search.fit(X_train, y_train)
print('XGB best:', xgb_search.best_params_, round(xgb_search.best_score_, 4))

XGB best: {'model__subsample': 0.7, 'model__n_estimators': 100, 'model__max_depth': 2, 'model__learning_rate': 0.1} 0.8489


In [17]:
import numpy as np
from sklearn.model_selection import cross_val_predict
from sklearn.metrics import precision_score, recall_score, f1_score

best_lr = lr_search.best_estimator_

proba = cross_val_predict(best_lr, X_train, y_train, cv=cv,
                          method='predict_proba')[:, 1]

print("threshold  precision  recall   f1    flagged%")
for t in np.arange(0.30, 0.76, 0.05):
    pred = (proba >= t).astype(int)
    print(f"  {t:.2f}      {precision_score(y_train, pred):.3f}     "
          f"{recall_score(y_train, pred):.3f}   {f1_score(y_train, pred):.3f}   "
          f"{pred.mean()*100:.1f}%")

threshold  precision  recall   f1    flagged%
  0.30      0.424     0.926   0.582   57.9%
  0.35      0.446     0.904   0.598   53.8%
  0.40      0.471     0.868   0.610   49.0%
  0.45      0.491     0.836   0.619   45.2%
  0.50      0.517     0.801   0.629   41.1%
  0.55      0.542     0.766   0.635   37.5%
  0.60      0.571     0.718   0.636   33.4%
  0.65      0.595     0.664   0.628   29.6%
  0.70      0.629     0.591   0.610   24.9%
  0.75      0.665     0.521   0.584   20.8%


In [18]:
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score

final_model = lr_search.best_estimator_   # already refit on all training data

test_proba = final_model.predict_proba(X_test)[:, 1]
test_pred = (test_proba >= 0.50).astype(int)

print("ROC-AUC:", round(roc_auc_score(y_test, test_proba), 3))
print(confusion_matrix(y_test, test_pred))
print(classification_report(y_test, test_pred, target_names=['Stayed', 'Churned']))

ROC-AUC: 0.841
[[746 289]
 [ 80 294]]
              precision    recall  f1-score   support

      Stayed       0.90      0.72      0.80      1035
     Churned       0.50      0.79      0.61       374

    accuracy                           0.74      1409
   macro avg       0.70      0.75      0.71      1409
weighted avg       0.80      0.74      0.75      1409



In [19]:
import pandas as pd

prep = final_model.named_steps['prep']
lr = final_model.named_steps['model']

coefs = pd.Series(lr.coef_[0], index=prep.get_feature_names_out())
coefs = coefs.sort_values()

print("Pushes AWAY from churn (lowest):")
print(coefs.head(8).round(2))
print("\nPushes TOWARD churn (highest):")
print(coefs.tail(8).round(2))

Pushes AWAY from churn (lowest):
cat__InternetService_No                        -2.81
num__MonthlyCharges                            -1.78
cat__Contract_Two year                         -1.25
num__tenure                                    -1.19
cat__Contract_One year                         -0.54
cat__InternetService_DSL                       -0.51
cat__PaymentMethod_Bank transfer (automatic)   -0.51
cat__PaymentMethod_Credit card (automatic)     -0.51
dtype: float64

Pushes TOWARD churn (highest):
cat__DeviceProtection_Yes           0.24
cat__PaperlessBilling_Yes           0.33
num__TotalCharges                   0.53
cat__MultipleLines_Yes              0.53
cat__PhoneService_Yes               0.65
cat__StreamingTV_Yes                0.80
cat__StreamingMovies_Yes            0.83
cat__InternetService_Fiber optic    1.73
dtype: float64


In [20]:
prep_alt = ColumnTransformer([
    ('num', StandardScaler(), ['tenure', 'TotalCharges']),
    ('cat', OneHotEncoder(drop='if_binary', handle_unknown='ignore',
                          sparse_output=False), cat_cols),
])

alt = Pipeline([('prep', prep_alt),
                ('model', LogisticRegression(max_iter=1000, class_weight='balanced', C=10))])

res = cross_validate(alt, X_train, y_train, cv=cv, scoring=['roc_auc', 'recall'])
print("AUC:", res['test_roc_auc'].mean().round(3), " Recall:", res['test_recall'].mean().round(3))

alt.fit(X_train, y_train)
c = pd.Series(alt.named_steps['model'].coef_[0],
              index=alt.named_steps['prep'].get_feature_names_out()).sort_values()
print(c.round(2))

AUC: 0.846  Recall: 0.804
num__tenure                                    -1.17
cat__InternetService_No                        -0.94
cat__Contract_Two year                         -0.85
cat__PhoneService_Yes                          -0.53
cat__OnlineSecurity_Yes                        -0.43
cat__TechSupport_Yes                           -0.36
cat__Dependents_Yes                            -0.23
cat__PaymentMethod_Credit card (automatic)     -0.22
cat__PaymentMethod_Bank transfer (automatic)   -0.21
cat__OnlineBackup_Yes                          -0.17
cat__PaymentMethod_Mailed check                -0.16
cat__Contract_One year                         -0.14
cat__InternetService_DSL                       -0.12
cat__DeviceProtection_Yes                      -0.05
cat__Partner_Yes                                0.02
cat__SeniorCitizen_1                            0.15
cat__PaymentMethod_Electronic check             0.19
cat__StreamingTV_Yes                            0.21
cat__MultipleLines_Y

In [21]:
prep_alt2 = ColumnTransformer([
    ('num', StandardScaler(), ['tenure', 'MonthlyCharges']),
    ('cat', OneHotEncoder(drop='if_binary', handle_unknown='ignore',
                          sparse_output=False), cat_cols),
])

alt2 = Pipeline([('prep', prep_alt2),
                 ('model', LogisticRegression(max_iter=1000, class_weight='balanced', C=10))])

res = cross_validate(alt2, X_train, y_train, cv=cv, scoring=['roc_auc', 'recall'])
print("AUC:", res['test_roc_auc'].mean().round(3), " Recall:", res['test_recall'].mean().round(3))

alt2.fit(X_train, y_train)
c2 = pd.Series(alt2.named_steps['model'].coef_[0],
               index=alt2.named_steps['prep'].get_feature_names_out()).sort_values()
print(c2.round(2))

AUC: 0.845  Recall: 0.793
cat__InternetService_No                        -2.72
num__MonthlyCharges                            -1.61
cat__Contract_Two year                         -1.24
num__tenure                                    -0.75
cat__InternetService_DSL                       -0.55
cat__Contract_One year                         -0.53
cat__PaymentMethod_Credit card (automatic)     -0.52
cat__PaymentMethod_Bank transfer (automatic)   -0.52
cat__PaymentMethod_Mailed check                -0.43
cat__Dependents_Yes                            -0.24
cat__OnlineSecurity_Yes                        -0.12
cat__PaymentMethod_Electronic check            -0.12
cat__TechSupport_Yes                           -0.05
cat__Partner_Yes                                0.02
cat__SeniorCitizen_1                            0.15
cat__OnlineBackup_Yes                           0.15
cat__Contract_Month-to-month                    0.20
cat__DeviceProtection_Yes                       0.26
cat__PaperlessBillin

In [22]:
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score

num_cols_f = ['tenure', 'MonthlyCharges']

prep_final = ColumnTransformer([
    ('num', StandardScaler(), num_cols_f),
    ('cat', OneHotEncoder(drop='if_binary', handle_unknown='ignore',
                          sparse_output=False), cat_cols),
])

final_model = Pipeline([
    ('prep', prep_final),
    ('model', LogisticRegression(max_iter=1000, class_weight='balanced', C=10)),
])

# Remove TotalCharges so the model expects exactly the 17 columns the app will send
X_train_f = X_train.drop(columns=['TotalCharges'])
X_test_f = X_test.drop(columns=['TotalCharges'])

final_model.fit(X_train_f, y_train)

test_proba = final_model.predict_proba(X_test_f)[:, 1]
test_pred = (test_proba >= 0.50).astype(int)

print("ROC-AUC:", round(roc_auc_score(y_test, test_proba), 3))
print(confusion_matrix(y_test, test_pred))
print(classification_report(y_test, test_pred, target_names=['Stayed', 'Churned']))

ROC-AUC: 0.838
[[754 281]
 [ 84 290]]
              precision    recall  f1-score   support

      Stayed       0.90      0.73      0.81      1035
     Churned       0.51      0.78      0.61       374

    accuracy                           0.74      1409
   macro avg       0.70      0.75      0.71      1409
weighted avg       0.80      0.74      0.75      1409



In [23]:
import os
import joblib

os.makedirs("../models", exist_ok=True)
joblib.dump(final_model, "../models/churn_model.joblib")

# Reload it as the app will, and predict for one made-up customer
loaded = joblib.load("../models/churn_model.joblib")

new_customer = pd.DataFrame([{
    'SeniorCitizen': 0, 'Partner': 'No', 'Dependents': 'No', 'tenure': 2,
    'PhoneService': 'Yes', 'MultipleLines': 'No', 'InternetService': 'Fiber optic',
    'OnlineSecurity': 'No', 'OnlineBackup': 'No', 'DeviceProtection': 'No',
    'TechSupport': 'No', 'StreamingTV': 'No', 'StreamingMovies': 'No',
    'Contract': 'Month-to-month', 'PaperlessBilling': 'Yes',
    'PaymentMethod': 'Electronic check', 'MonthlyCharges': 80.0
}])

prob = loaded.predict_proba(new_customer)[0, 1]
print(f"Churn probability: {prob:.2f}")

Churn probability: 0.78


In [24]:
safe_customer = new_customer.copy()
safe_customer.loc[0, ['tenure', 'Contract', 'InternetService', 'PaymentMethod',
                      'OnlineSecurity', 'TechSupport', 'MonthlyCharges']] = \
    [60, 'Two year', 'DSL', 'Credit card (automatic)', 'Yes', 'Yes', 55.0]

print(f"Churn probability: {loaded.predict_proba(safe_customer)[0, 1]:.2f}")

Churn probability: 0.03
